# Exploración de Distribuciones y Data Profiling - diabetes_readmission

Este notebook realiza un análisis comparativo entre las particiones In-Distribution (Train) y Out-of-Distribution (OOD Test) del dataset `diabetes_readmission` usando `data_profiling` (anteriormente `ydata-profiling`).


In [ ]:
from pathlib import Path
import pandas as pd
from tableshift import get_dataset
from tableshift.core.features import PreprocessorConfig
from data_profiling import ProfileReport

In [ ]:
def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise RuntimeError("No se encontró la raíz del repo")


PROJECT_ROOT = find_project_root()
CACHE_DIR = PROJECT_ROOT / "data" / "raw" / "tableshift_cache"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ydata_profiling" / "diabetes_readmission"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache dir: {CACHE_DIR}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
def passthrough_preprocessor() -> PreprocessorConfig:
    return PreprocessorConfig(
        categorical_features="passthrough",
        numeric_features="passthrough",
        dropna="all",
    )

In [ ]:
print("Cargando dataset diabetes_readmission...")
dset = get_dataset(
    "diabetes_readmission",
    cache_dir=str(CACHE_DIR),
    preprocessor_config=passthrough_preprocessor(),
)
print("¡Cargado!")

In [ ]:
X_train, y_train, _, _ = dset.get_pandas(split="train")
X_ood, y_ood, _, _ = dset.get_pandas(split="ood_test")

print(f"Train (ID): {X_train.shape}")
print(f"OOD Test  : {X_ood.shape}")

In [ ]:
df_train = X_train.copy()
df_train["readmitted"] = y_train

df_ood = X_ood.copy()
df_ood["readmitted"] = y_ood

print("Generando reporte de Train (ID) para diabetes_readmission...")
report_train = ProfileReport(
    df_train, title=f"{dataset_name} - Train Split (ID)", minimal=True
)

print("Generando reporte de OOD Test para diabetes_readmission...")
report_ood = ProfileReport(
    df_ood, title=f"{dataset_name} - OOD Test Split", minimal=True
)

print("Guardando reportes individuales...")
report_train.to_file(OUTPUT_DIR / "report_train.html")
report_ood.to_file(OUTPUT_DIR / "report_ood.html")

print("Generando y guardando reporte comparativo (Train vs OOD)... ")
comparison_report = report_train.compare(report_ood)
comparison_report.to_file(OUTPUT_DIR / "report_comparison.html")
print("¡Listo! Reportes guardados en:", OUTPUT_DIR)